# Gold + serving — Google rating history (fact_place_rating_snapshot, agg_restaurant_rating_daily)

Added 2026-11-01, reworked 2026-11-02: Silver keeps every Google read (`stg_places`: Text Search, `stg_restaurant_place_details`: Place Details), so the separate `stg_place_rating_reads` table is gone.

| Table | Layer | Grain | Load |
| --- | --- | --- | --- |
| `fact_place_rating_snapshot` | Gold, periodic snapshot at read grain | `place_id` × `snapshot_ts` × `source` (`place_details` / `text_search`) | full rebuild (small) |
| `agg_restaurant_rating_daily` | Serving | place × Vietnam day, **dense** from the first read to today (last read carried forward) | full rebuild |

**fact_place_rating_snapshot:** every read with a rating; `restaurant_key` = current `dim_restaurant` version with that `place_id` (−1 if none); `rating_change` / `rating_count_gained` vs the previous read of the same place (any source; additive over reads, NULL on the first read). `rating` / `rating_count` semi-additive.

**agg_restaurant_rating_daily** (same columns as the old `fact_restaurant_rating_daily_snapshot`): `rating`, `rating_count` of the latest read on or before the day (semi-additive), `rating_change` / `rating_count_gained` vs the previous day (non-zero only on a day with a new read, covering the whole gap since the read before; additive over days, NULL on the first day), `is_new_read`, `days_since_read`, `read_at`, `read_source`.

Reads per place (900 Place Details calls / month): restaurants with a YouTube review every 30 days, top 20 by `legit_score` every 7 days, Google-only places every 90 days, plus every manual Text Search. Google's rating is the all-time average, so it moves slowly; `rating_count_gained` is usually the more telling number.

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
PLACES = f"{SILVER}.stg_places"                       # every Text Search read
DETAILS = f"{SILVER}.stg_restaurant_place_details"    # every Place Details read
FACT = f"{GOLD}.fact_place_rating_snapshot"
DAILY = f"{GOLD}.agg_restaurant_rating_daily"
print(f"Silver: {PLACES}, {DETAILS} · Gold: {FACT} · Serving: {DAILY}")

In [ ]:
reads = []
if spark.catalog.tableExists(DETAILS):
    reads.append(f"""SELECT place_id, fetched_at AS snapshot_ts, 'place_details' AS source,
                            CAST(rating AS DOUBLE) AS rating, CAST(rating_count AS INT) AS rating_count
                     FROM {DETAILS} WHERE fetch_status = 'ok' AND rating IS NOT NULL AND fetched_at IS NOT NULL""")
if spark.catalog.tableExists(PLACES) and "snapshot_ts" in spark.table(PLACES).columns:
    reads.append(f"""SELECT place_id, snapshot_ts, 'text_search' AS source, rating, rating_count
                     FROM {PLACES} WHERE rating IS NOT NULL""")
reads_sql = " UNION ALL ".join(reads) if reads else (
    "SELECT CAST(NULL AS STRING) AS place_id, CAST(NULL AS TIMESTAMP) AS snapshot_ts, CAST(NULL AS STRING) AS source, "
    "CAST(NULL AS DOUBLE) AS rating, CAST(NULL AS INT) AS rating_count WHERE false")
spark.sql(f"""
    CREATE OR REPLACE TABLE {FACT}
    COMMENT 'Periodic snapshot at read grain: 1 Google read of a place (place_id x snapshot_ts x source). rating / rating_count semi-additive, rating_change / rating_count_gained (vs previous read) additive.'
    AS
    WITH r AS (
        SELECT place_id, snapshot_ts, source, max(rating) AS rating, max(rating_count) AS rating_count
        FROM ({reads_sql}) WHERE place_id IS NOT NULL GROUP BY place_id, snapshot_ts, source
    ),
    restaurant AS (
        SELECT place_id, min(restaurant_key) AS restaurant_key
        FROM {GOLD}.dim_restaurant WHERE is_current AND place_id IS NOT NULL GROUP BY place_id
    )
    SELECT
        r.snapshot_ts,
        CAST(date_format(from_utc_timestamp(r.snapshot_ts, '{SNAPSHOT_TZ}'), 'yyyyMMdd') AS INT) AS snapshot_date_key,
        COALESCE(d.restaurant_key, -1)                                 AS restaurant_key,
        r.place_id, r.source, r.rating, r.rating_count,
        ROUND(r.rating - LAG(r.rating) OVER w, 2)                      AS rating_change,
        r.rating_count - LAG(r.rating_count) OVER w                    AS rating_count_gained,
        LAG(r.snapshot_ts) OVER w IS NULL                              AS is_first_snapshot,
        current_timestamp()                                            AS etl_date
    FROM r LEFT JOIN restaurant d ON d.place_id = r.place_id
    WINDOW w AS (PARTITION BY r.place_id ORDER BY r.snapshot_ts, r.source)
""")
s = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT place_id, snapshot_ts, source) AS grain, COUNT(DISTINCT place_id) AS places FROM {FACT}").collect()[0]
print(f"✅ {FACT}: {s['n']} reads of {s['places']} places · grain: " + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
if s["n"] != s["grain"]:
    raise AssertionError(f"{FACT}: duplicate (place_id, snapshot_ts, source) rows")

In [ ]:
TODAY = f"to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}'))"
spark.sql(f"""
    CREATE OR REPLACE TABLE {DAILY}
    COMMENT 'Serving: 1 Google place (restaurant) x 1 Vietnam day, dense (last read carried forward), from fact_place_rating_snapshot. rating / rating_count semi-additive, rating_change / rating_count_gained additive over days.'
    AS
    WITH day_reads AS (
        SELECT place_id, to_date(from_utc_timestamp(snapshot_ts, '{SNAPSHOT_TZ}')) AS read_date,
               max_by(named_struct('rating', rating, 'rating_count', rating_count, 'read_at', snapshot_ts,
                                   'source', source, 'restaurant_key', restaurant_key), snapshot_ts) AS r
        FROM {FACT}
        GROUP BY place_id, to_date(from_utc_timestamp(snapshot_ts, '{SNAPSHOT_TZ}'))
    ),
    days AS (
        SELECT place_id, explode(sequence(MIN(read_date), {TODAY})) AS snapshot_date
        FROM day_reads GROUP BY place_id
    ),
    filled AS (
        SELECT d.place_id, d.snapshot_date, r.r IS NOT NULL AS is_new_read,
               last_value(r.r, true) OVER (PARTITION BY d.place_id ORDER BY d.snapshot_date
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS r
        FROM days d
        LEFT JOIN day_reads r ON r.place_id = d.place_id AND r.read_date = d.snapshot_date
    ),
    restaurant AS (
        SELECT place_id, min(restaurant_key) AS restaurant_key
        FROM {GOLD}.dim_restaurant WHERE is_current AND place_id IS NOT NULL GROUP BY place_id
    )
    SELECT
        CAST(date_format(f.snapshot_date, 'yyyyMMdd') AS INT)              AS snapshot_date_key,
        COALESCE(dr.restaurant_key, -1)                                    AS restaurant_key,
        f.place_id,
        f.r.rating                                                         AS rating,
        f.r.rating_count                                                   AS rating_count,
        ROUND(f.r.rating - LAG(f.r.rating) OVER w, 2)                      AS rating_change,
        f.r.rating_count - LAG(f.r.rating_count) OVER w                    AS rating_count_gained,
        f.is_new_read,
        LAG(f.snapshot_date) OVER w IS NULL                                AS is_first_snapshot,
        datediff(f.snapshot_date, to_date(from_utc_timestamp(f.r.read_at, '{SNAPSHOT_TZ}'))) AS days_since_read,
        f.r.read_at                                                        AS read_at,
        f.r.source                                                         AS read_source,
        current_timestamp()                                                AS etl_date
    FROM filled f
    LEFT JOIN restaurant dr ON dr.place_id = f.place_id
    WINDOW w AS (PARTITION BY f.place_id ORDER BY f.snapshot_date)
""")

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id, snapshot_date_key) AS grain, COUNT(DISTINCT place_id) AS places,
           COUNT_IF(restaurant_key = -1) AS unknown_restaurant, MAX(snapshot_date_key) AS last_day,
           COUNT_IF(rating_change <> 0) AS rating_changes
    FROM {DAILY}
""").collect()[0]
print(f"📊 {DAILY}: {s['n']} rows · {s['places']} places · last day {s['last_day']} · {s['rating_changes']} rating changes · "
      f"{s['unknown_restaurant']} rows without a restaurant (key -1) · grain (place_id x day): " + ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
if s["n"] != s["grain"]:
    raise AssertionError(f"{DAILY}: duplicate (place_id, day) rows")
display(spark.sql(f"""
    SELECT place_id, snapshot_ts, source, rating, rating_change, rating_count, rating_count_gained
    FROM {FACT} WHERE NOT is_first_snapshot ORDER BY abs(rating_change) DESC, rating_count_gained DESC LIMIT 20
"""))
print("\n--- place_rating complete ---")